<a href="https://colab.research.google.com/github/HowardChen0211/dataManPy/blob/main/ProblemSet_2_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

# California Housing Market, 2020–2025: Home Values, Rents, and Local Labor Markets

### Research questions
1. Which California counties saw the fastest growth in home values from 2020 to 2025?
2. To what extent is home-value growth associated with rent growth across counties?
3. Are county size (population) and the unemployment rate related to how fast housing costs grew?


### Why these data
| Dataset | What it gives us | Why it fits | Data format |
|---|---|---|---|
| Zillow Home Value Index (ZHVI), county, monthly | home value (35th–65th percentile homes) | Consistent monthly series for all 58 CA counties | .csv |
| Zillow Observed Rent Index (ZORI), county, monthly | rent index | Same source and geography as ZHVI, so the two can be matched county by county | .csv |
| U.S. Census Bureau, Population Division. | Total Population (July 1, 2025) | Measures county size; has a FIPS code to merge on | .csv |
| ACS 2020–2024 5-year estimates (via Social Explorer) | Labor force and unemployed, by county | Labor-market condition over roughly the same window as the housing data | .csv |

#### Unit of analysis
California counties (n = 58). The merge key across all datasets is the 5-digit county **FIPS code**.


## Import Libraries

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#will display all output not just last command
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.float_format', '{:.3f}'.format) # Turn off the scientific notation https://medium.com/@amit25173/steps-to-pandas-turn-off-scientific-notation-62c44b86ee24

## Load the data

In [2]:
# Zillow Home Value Index (ZHVI)
zhvi = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv")

# Zillow Observed Rent Index (ZORI)
zori = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/County_zori_uc_sfrcondomfr_sm_month.csv")

# County population 2025
population = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_Population_estimates_2025.csv")

# ACS 2020-2024 5-year: civilian labor force 16 and over,
unemployment = pd.read_csv("https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/CA_UnemploymentRate.csv")

In [3]:
# take a look about the data (first 5 rows)
zhvi.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2000-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,210217.606,...,876483.587,880007.554,881688.904,881664.430,879853.110,877155.644,874790.007,872663.237,872220.754,873008.422
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,152726.258,...,327258.568,328895.004,330363.857,332095.628,333712.156,335053.112,336017.428,337263.419,339162.650,341569.751
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,113021.492,...,283353.950,283400.843,283252.913,282917.698,282310.099,281596.697,280609.690,279554.676,278797.692,278517.967
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,146106.504,...,459582.164,460367.020,461220.421,461894.031,461740.422,460735.658,459147.558,457589.894,456733.110,456252.051
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,216416.160,...,925424.341,927598.249,928828.819,929848.898,930228.330,930115.693,929543.266,928860.114,929330.518,930799.655


In [4]:
# take a look about the data (first 5 rows)
zori.head()

,RegionID,SizeRank,RegionName,RegionType,StateName,State,Metro,StateCodeFIPS,MunicipalCodeFIPS,2015-01-31,...,2025-11-30,2025-12-31,2026-01-31,2026-02-28,2026-03-31,2026-04-30,2026-05-31,2026-06-30,2026-07-31,2026-08-31
0,3101,0,Los Angeles County,county,CA,CA,"Los Angeles-Long Beach-Anaheim, CA",6,37,1691.320,...,2777.262,2768.420,2770.406,2776.600,2788.838,2797.135,2805.344,2809.822,2810.309,2813.614
1,139,1,Cook County,county,IL,IL,"Chicago-Naperville-Elgin, IL-IN-WI",17,31,1413.147,...,2145.212,2143.577,2157.834,2178.380,2204.530,2226.641,2248.019,2265.658,2274.611,2272.117
2,1090,2,Harris County,county,TX,TX,"Houston-The Woodlands-Sugar Land, TX",48,201,1171.144,...,1577.797,1572.948,1571.596,1569.418,1570.920,1573.802,1580.332,1584.251,1586.397,1586.095
3,2402,3,Maricopa County,county,AZ,AZ,"Phoenix-Mesa-Chandler, AZ",4,13,905.494,...,1692.043,1689.654,1690.670,1697.466,1702.306,1706.189,1711.234,1712.111,1715.836,1716.663
4,2841,4,San Diego County,county,CA,CA,"San Diego-Chula Vista-Carlsbad, CA",6,73,1628.319,...,2917.657,2912.131,2915.513,2929.541,2941.689,2951.787,2963.444,2976.321,2984.436,2993.632


In [5]:
# take a look about the data (first 5 rows)
population.head()

,FIPS Code,Summary Level,Area Name,Qualified Area Name,"Total Population (July 1, 2025)"
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,ORG_T007_001
1,06001,SL050,Alameda County,"Alameda County, California",1636630
2,06003,SL050,Alpine County,"Alpine County, California",1043
3,06005,SL050,Amador County,"Amador County, California",41876
4,06007,SL050,Butte County,"Butte County, California",209211


In [6]:
# take a look about the data (first 5 rows)
unemployment.head()

,FIPS,GeoLevel,NAME,Qualified Area Name,Area (Land),Area (Water),Civilian Population in Labor Force 16 Years and Over,Civilian Population in Labor Force 16 Years and Over: Employed,Civilian Population in Labor Force 16 Years and Over: Unemployed
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,Geo_AREALAND,Geo_AREAWATER,SE_A17005_001,SE_A17005_002,SE_A17005_003
1,06001,SL050,"Alameda County, California","Alameda County, California",1910010500,216909647,908995,860129,48866
2,06003,SL050,"Alpine County, California","Alpine County, California",1912292607,12557304,837,800,37
3,06005,SL050,"Amador County, California","Amador County, California",1539967082,29437117,17257,16325,932
4,06007,SL050,"Butte County, California","Butte County, California",4238545149,105204062,100743,93228,7515


## First look at the data

In [7]:
# prints information about a DataFrame
print(zhvi.info(), "\n")
print(zori.info(), "\n")
print(population.info(), "\n")
print(unemployment.info(), "\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3071 entries, 0 to 3070
Columns: 329 entries, RegionID to 2026-08-31
dtypes: float64(320), int64(4), object(5)
memory usage: 7.7+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1407 entries, 0 to 1406
Columns: 149 entries, RegionID to 2026-08-31
dtypes: float64(140), int64(4), object(5)
memory usage: 1.6+ MB
None 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   FIPS Code                        59 non-null     object
 1   Summary Level                    59 non-null     object
 2   Area Name                        59 non-null     object
 3   Qualified Area Name              59 non-null     object
 4   Total Population (July 1, 2025)  59 non-null     object
dtypes: object(5)
memory usage: 2.4+ KB
None 

<class 'pandas.core.frame.DataFrame'>
RangeI

In [8]:
# See how many columns in the data
print("Home Index: ", len(zhvi.columns))
print("Rent Index: ", len(zori.columns))
print("Population: ", len(population.columns))
print("Unemployment: ", len(unemployment.columns))

Home Index:  329
Rent Index:  149
Population:  5
Unemployment:  9


In [9]:
# See how big the data
print(zhvi.shape) # (3071, 329)
print(zori.shape) # (1407, 149)
print(population.shape) # (59, 5)
print(unemployment.shape) # (59, 9)

(3071, 329)
(1407, 149)
(59, 5)
(59, 9)


In [10]:
# Missing values
print(zhvi.isna().sum())
print()
print(zori.isna().sum())
print()
print(population.isna().sum())
print()
print(unemployment.isna().sum())

RegionID      0
SizeRank      0
RegionName    0
RegionType    0
StateName     0
             ..
2026-04-30    0
2026-05-31    0
2026-06-30    0
2026-07-31    0
2026-08-31    0
Length: 329, dtype: int64

RegionID        0
SizeRank        0
RegionName      0
RegionType      0
StateName       0
             ... 
2026-04-30    224
2026-05-31    196
2026-06-30    169
2026-07-31    114
2026-08-31      1
Length: 149, dtype: int64

FIPS Code                          0
Summary Level                      0
Area Name                          0
Qualified Area Name                0
Total Population (July 1, 2025)    0
dtype: int64

FIPS                                                                0
GeoLevel                                                            0
NAME                                                                0
Qualified Area Name                                                 0
Area (Land)                                                         0
Area (Water)           

In [11]:
# Check Duplicates
print(zhvi.duplicated().sum())
print(zori.duplicated().sum())
print(population.duplicated().sum())
print(unemployment.duplicated().sum())

0
0
0
0


# ProblemSet 1

## Clean and reshape

Transform both zillow dataset into long format *ZHVI* & *ZORI* and filtered the dates to include only records from 2020 to 2025.

In [12]:
START, END = "2020-01", "2026-01"

def zillow_to_long(data, value_name, start=START, end=END):

    cols = ["RegionID", "SizeRank", "RegionName", "RegionType", "StateName", "StateCodeFIPS", "MunicipalCodeFIPS"]
    dates = [i for i in data.columns if i[0].isdigit() and start <= i < end]

    ca = data.loc[data["StateName"] == "CA", cols + dates].copy()
    # FIPS codes is 00+000
    ca["FIPS"] = (ca["StateCodeFIPS"].astype(str).str.zfill(2)
                  + ca["MunicipalCodeFIPS"].astype(str).str.zfill(3))

    ca["RegionName"] = ca["RegionName"].str.strip()

    # wide to long https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    df = ca.melt(id_vars=["FIPS", "RegionName"], value_vars=dates,
                   var_name="Date", value_name=value_name)

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

    return df.sort_values(["FIPS", "Date"]).reset_index(drop=True)

# Call the function
zhvi_CA = zillow_to_long(zhvi, "ZHVI")
zori_CA = zillow_to_long(zori, "ZORI")

# See the shape & check unique values
print("ZHVI:", zhvi_CA.shape, "| counties:", zhvi_CA["FIPS"].nunique(), "| months:", zhvi_CA["Date"].nunique())
print("ZORI:", zori_CA.shape, "| counties:", zori_CA["FIPS"].nunique(), "| months:", zori_CA["Date"].nunique())

ZHVI: (4176, 4) | counties: 58 | months: 72
ZORI: (3528, 4) | counties: 49 | months: 72


ZHVI covers all 58 counties × 72 months = 4,176 rows. ZORI has fewer counties because Zillow does not publish rents for some small rural counties.

In [13]:
# See the data types in the dataframe
zhvi_CA.dtypes

,0
FIPS,object
RegionName,object
Date,datetime64[ns]
ZHVI,float64


In [14]:
zori_CA.dtypes

,0
FIPS,object
RegionName,object
Date,datetime64[ns]
ZORI,float64


In [15]:
zhvi_CA.head()

,FIPS,RegionName,Date,ZHVI
0,06001,Alameda County,2020-01-31,842261.950
1,06001,Alameda County,2020-02-29,845568.279
2,06001,Alameda County,2020-03-31,849961.749
3,06001,Alameda County,2020-04-30,854051.748
4,06001,Alameda County,2020-05-31,853204.560


In [16]:
zori_CA.head()

,FIPS,RegionName,Date,ZORI
0,06001,Alameda County,2020-01-31,2547.423
1,06001,Alameda County,2020-02-29,2548.081
2,06001,Alameda County,2020-03-31,2563.948
3,06001,Alameda County,2020-04-30,2561.371
4,06001,Alameda County,2020-05-31,2556.497


### Merge 1: ZHVI × ZORI housing

In [17]:
housing = zhvi_CA.merge(
    zori_CA.drop(columns="RegionName"),
    on=["FIPS", "Date"],
    how="outer",            # keep every county-month that has a home value
    validate="1:1",
    indicator=True,
)

In [18]:
housing.query("_merge == 'left_only'")

,FIPS,RegionName,Date,ZHVI,ZORI,_merge
72,06003,Alpine County,2020-01-31,389374.972,NaN,left_only
73,06003,Alpine County,2020-02-29,389221.795,NaN,left_only
74,06003,Alpine County,2020-03-31,388438.388,NaN,left_only
75,06003,Alpine County,2020-04-30,388918.625,NaN,left_only
76,06003,Alpine County,2020-05-31,389024.193,NaN,left_only
...,...,...,...,...,...,...
3811,06105,Trinity County,2025-08-31,263132.164,NaN,left_only
3812,06105,Trinity County,2025-09-30,262568.495,NaN,left_only
3813,06105,Trinity County,2025-10-31,261993.268,NaN,left_only
3814,06105,Trinity County,2025-11-30,261540.047,NaN,left_only


In [19]:
# checks
assert len(housing) == len(zhvi_CA) == 58 * 72
assert housing[["FIPS", "Date"]].duplicated().sum() == 0

print(housing["_merge"].value_counts(), "\n")

_merge
both          3528
left_only      648
right_only       0
Name: count, dtype: int64 



In [20]:
no_rent = sorted(housing.loc[housing["_merge"] == "left_only", "RegionName"].unique())
print(f"{len(no_rent)} counties have no ZORI in any month:\n", no_rent)
housing = housing.drop(columns="_merge")

9 counties have no ZORI in any month:
 ['Alpine County', 'Colusa County', 'Lassen County', 'Mariposa County', 'Modoc County', 'Mono County', 'Plumas County', 'Sierra County', 'Trinity County']


**Interpretation.** The left join keeps all 4,176 county-months (58 x 72), so no home-value observation is lost. The 648 `left_only` rows are the 9 counties Zillow does not publish rents for at all (Alpine, Colusa, Lassen, Mariposa, Modoc, Mono, Plumas, Sierra, Trinity): small rural counties with too few rental listings for a stable index. That is expected and is not a merge error.

In [21]:
# Replace on condition
housing["rent_data_status"] = "Available"

housing.loc[housing["ZORI"].isna(), "rent_data_status"] = "No ZORI data"

housing["rent_data_status"].value_counts()

,count
rent_data_status,
Available,2691
No ZORI data,1485


### Merge 2: Labor-market data

#### Population (Social Explorer)

In [22]:
population.head()

,FIPS Code,Summary Level,Area Name,Qualified Area Name,"Total Population (July 1, 2025)"
0,Geo_FIPS,_geo_level_,Geo_NAME,Geo_qname,ORG_T007_001
1,06001,SL050,Alameda County,"Alameda County, California",1636630
2,06003,SL050,Alpine County,"Alpine County, California",1043
3,06005,SL050,Amador County,"Amador County, California",41876
4,06007,SL050,Butte County,"Butte County, California",209211


In [23]:
population.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 59 entries, 0 to 58
Data columns (total 5 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   FIPS Code                        59 non-null     object
 1   Summary Level                    59 non-null     object
 2   Area Name                        59 non-null     object
 3   Qualified Area Name              59 non-null     object
 4   Total Population (July 1, 2025)  59 non-null     object
dtypes: object(5)
memory usage: 2.4+ KB


In [24]:
population = (population.iloc[1:,:][["FIPS Code", "Area Name", "Total Population (July 1, 2025)"]].rename(columns={"FIPS Code": "FIPS", "Area Name": "County", "Total Population (July 1, 2025)": "pop_2025"}))

print(population.shape)

population.head()

(58, 3)


,FIPS,County,pop_2025
1,06001,Alameda County,1636630
2,06003,Alpine County,1043
3,06005,Amador County,41876
4,06007,Butte County,209211
5,06009,Calaveras County,46605


In [25]:
population["pop_2025"] = population["pop_2025"].astype(int)

#### Unemployment (ACS 2020–2024 5-year)

In [26]:
# row 0 holds the Social Explorer variable codes (Geo_FIPS, SE_A17005_001, ...), so drop it
unemployment = (unemployment
                .loc[1:, ["FIPS", "NAME",
                          "Civilian Population in Labor Force 16 Years and Over",
                          "Civilian Population in Labor Force 16 Years and Over: Unemployed"]]
                .rename(columns={"NAME": "County",
                                 "Civilian Population in Labor Force 16 Years and Over": "labor_force",
                                 "Civilian Population in Labor Force 16 Years and Over: Unemployed": "unemployed"}))

unemployment["County"] = unemployment["County"].str.replace(", California", "")

unemployment["FIPS"] = unemployment["FIPS"].astype(str).str.zfill(5)

unemployment[["labor_force", "unemployed"]] = unemployment[["labor_force", "unemployed"]].astype(int)

unemployment["unemp_rate"] = unemployment["unemployed"] / unemployment["labor_force"] * 100

print(unemployment.shape)

unemployment.head(3)

(58, 5)


,FIPS,County,labor_force,unemployed,unemp_rate
1,06001,Alameda County,908995,48866,5.376
2,06003,Alpine County,837,37,4.421
3,06005,Amador County,17257,932,5.401


#### Merge Population x Unemployment

In [27]:
labor = unemployment.merge(population.drop(columns="County"), on="FIPS",
                           how="outer", validate="1:1", indicator=True)

print(labor["_merge"].value_counts())

labor = labor.drop(columns="_merge")

labor.head()

_merge
both          58
left_only      0
right_only     0
Name: count, dtype: int64


,FIPS,County,labor_force,unemployed,unemp_rate,pop_2025
0,06001,Alameda County,908995,48866,5.376,1636630
1,06003,Alpine County,837,37,4.421,1043
2,06005,Amador County,17257,932,5.401,41876
3,06007,Butte County,100743,7515,7.460,209211
4,06009,Calaveras County,19422,877,4.515,46605


In [28]:
labor.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   FIPS         58 non-null     object 
 1   County       58 non-null     object 
 2   labor_force  58 non-null     int64  
 3   unemployed   58 non-null     int64  
 4   unemp_rate   58 non-null     float64
 5   pop_2025     58 non-null     int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 2.8+ KB


In [29]:
# Replace on condition

labor["unemployment_status"] = "Low"

labor.loc[labor["unemp_rate"] >= 5,"unemployment_status"] = "Moderate"

labor.loc[labor["unemp_rate"] >= 8,"unemployment_status"] = "High"

labor["unemployment_status"].value_counts()

,count
unemployment_status,
Moderate,33
High,17
Low,8


### Mapping: #1. counties to regions
Group into Six Regions

Source: https://coolcalifornia.arb.ca.gov/regions

In [30]:
regions = {
    "Bay Area": ["Alameda", "Contra Costa", "Marin", "Napa", "San Francisco",
                 "San Mateo", "Santa Clara", "Solano", "Sonoma"],
    "Southern California": ["Imperial", "Los Angeles", "Orange", "Riverside",
                            "San Bernardino", "San Diego", "Ventura"],
    "Central Coast": ["Monterey", "San Benito", "San Luis Obispo", "Santa Barbara", "Santa Cruz"],
    "Central Valley": ["Butte", "Colusa", "Fresno", "Glenn", "Kern", "Kings", "Madera",
                       "Merced", "Sacramento", "San Joaquin", "Stanislaus", "Sutter",
                       "Tehama", "Tulare", "Yolo", "Yuba"],
    "Sierra / Gold Country": ["Alpine", "Amador", "Calaveras", "El Dorado", "Inyo",
                              "Mariposa", "Mono", "Nevada", "Placer", "Plumas",
                              "Sierra", "Tuolumne"],
    "North Coast / Far North": ["Del Norte", "Humboldt", "Lake", "Lassen", "Mendocino",
                                "Modoc", "Shasta", "Siskiyou", "Trinity"],
}

county_to_region = {}

for region, county in regions.items():
  for c in county:
    county_to_region[f"{c} County"] = region

housing["region"] = housing["RegionName"].map(county_to_region)

housing.drop_duplicates("FIPS")["region"].value_counts() # Total should be 58

,count
region,
Central Valley,16
Sierra / Gold Country,12
Bay Area,9
North Coast / Far North,9
Southern California,7
Central Coast,5


### Mapping #2: population to county size

In [31]:
labor["size_tier"] = pd.cut(labor["pop_2025"],
                            bins=[0, 100_000, 1_000_000, np.inf],
                            labels=["Small (<100k)", "Mid (100k-1M)", "Large (1M+)"])
labor["size_tier"].value_counts().sort_index()

,count
size_tier,
Small (<100k),22
Mid (100k-1M),26
Large (1M+),10


# ProblemSet 2

## Reference

1. Zillow Home Value Index (ZHVI), county, all homes, mid tier (35th–65th pct), smoothed, seasonally adjusted, monthly: https://files.zillowstatic.com/research/public_csvs/zhvi/County_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv
   (Zillow Research data page: https://www.zillow.com/research/data/ → *Home Values* → Data Type "ZHVI All Homes (SFR, Condo/Co-op) Time Series, Smoothed, Seasonally Adjusted", Geography "County")
2. Zillow Observed Rent Index (ZORI), county, all homes plus multifamily, smoothed, monthly: https://files.zillowstatic.com/research/public_csvs/zori/County_zori_uc_sfrcondomfr_sm_month.csv
   (same page → *Rentals* → "ZORI (Smoothed): All Homes Plus Multifamily Time Series", Geography "County")
3. Wikipedia, *List of counties in California* (2025 population column): https://en.wikipedia.org/wiki/List_of_counties_in_California — my cleaned copy: https://github.com/HowardChen0211/dataManPy/raw/refs/heads/main/data/population_CA.csv
4. U.S. Census Bureau, American Community Survey 2020-2024 5-Year Estimates, table A17005 (Employment Status for Total Population 16 Years and Over), CA counties, via Social Explorer: https://www.socialexplorer.com/reports/socialexplorer/en/report/bac15980-b308-11f1-9512-c38da70940e6 — exported file: https://github.com/HowardChen0211/gisPy/raw/refs/heads/main/CA_UnemploymentRate.csv
5. California Air Resources Board, Cool California regions (county-to-region grouping): https://coolcalifornia.arb.ca.gov/regions
